In [ ]:
import requests
from time import sleep
from tqdm import tqdm

from bs4 import BeautifulSoup
import json

import polars as pl

In [46]:
s = requests.Session()

headers = {
    "accept": "application/json",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "ocp-apim-subscription-key": "5a7825734ee9435191bcbad10177d750",
    "origin": "https://countryhistoricalprofiles.worldbank.org",
    "referer": "https://countryhistoricalprofiles.worldbank.org/",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7,es-ES;q=0.6,es;q=0.5"
}

s = requests.Session()
s.headers.update(headers)

In [83]:
url = "https://webapi.worldbank.org/wbg/archives/countries"

countries_get = s.get(url)

if countries_get.status_code != 200:
    raise Exception("Failed to fetch countries")

countries = json.loads(countries_get.content)

In [84]:
reports, errors = [], []

with tqdm(countries, desc="Processing countries") as pbar:
    for country in countries:
        sleep(1) # Consider increasing it
        url = "https://webapi.worldbank.org/wbg/archives/" + country["countryID"]

        country_data_get = s.get(url)

        try:
            if country_data_get.status_code != 200:
                raise Exception(f"Bad status code: {country_data_get.status_code}")

            country_data_raw = json.loads(country_data_get.content)
            soup = BeautifulSoup(
                country_data_raw[country["countryName"]][0]["dandr"]["earlyCountryEconomicReport"],
                "html.parser"
            )

            for tr in soup.find_all("tr"):
                reports.append({
                    "countryName": country["countryName"],
                    "countryID": country["countryID"],
                    "title": tr.find("a").get_text(),
                    "link": tr.find("a")["href"],
                    "date": tr.find_all("p")[1].get_text()
                })
        except Exception as e:
            errors.append({
                "countryName": country["countryName"],
                "countryID": country["countryID"],
                "error": str(e)
            })

        pbar.update(1)

In [ ]:
reports

[{'countryName': 'Afghanistan',
  'countryID': 'AFG',
  'title': 'Afghanistan - The development prospects',
  'link': 'https://documents.worldbank.org/en/publication/documents-reports/documentdetail/270251467989969662/Afghanistan-The-development-prospects',
  'date': 'February 28, 1963'},
 {'countryName': 'Afghanistan',
  'countryID': 'AFG',
  'title': 'Afghanistan - Economic development program',
  'link': 'https://documents.worldbank.org/en/publication/documents-reports/documentdetail/540181467995632715/Afghanistan-Economic-development-program',
  'date': 'January 31, 1967'},
 {'countryName': 'Afghanistan',
  'countryID': 'AFG',
  'title': 'Afghanistan - Current economic position and prospects',
  'link': 'https://documents.worldbank.org/en/publication/documents-reports/documentdetail/325641468184761990/Afghanistan-Current-economic-position-and-prospects',
  'date': 'May 31, 1969'},
 {'countryName': 'Afghanistan',
  'countryID': 'AFG',
  'title': 'Afghanistan - Current economic posit

In [88]:
[c["countryName"] for c in errors]
errors[3]["error"]

'Expecting value: line 1 column 1 (char 0)'

In [93]:
df = pl.DataFrame(reports).with_columns([
    pl.col("date").str.strptime(pl.Date, "%B %d, %Y", strict=False),
    pl.col("title").str.strip_chars().alias("title")
]).with_columns([
    pl.col("date").dt.year().alias("year")
])

with open("wb_country_year.csv", "w", encoding="utf-8") as f:
    df.write_csv(f)